# Donor Modeling Pipeline v2.0

Production script. For each entry in `jobruns.yaml`, it trains and selects a model per cohort, scores the OOT frame, and publishes the model, its spec and the scores. It runs unattended: the inputs are the train and OOT feature CSVs plus three YAML configs.

**Running it**
- Paths, TRAIN_FILE and OOT_FILE are set in 1. Run settings. With TRAIN_FILE = None, the newest train_features_t<date>_*.csv in Constructed Data is used; with OOT_FILE = None, its paired oot_features_*.csv. Both files are read once and shared by every label.
- Labels and cohorts are independent: one that fails is logged and the rest continue. The final cell raises if anything failed, so a scheduler sees it.

**`jobruns.yaml`** (in the config folder) lists what to run:
```yaml
runs:
  - label: label_will_start_monthly_next_365d
    cohorts: [all_donors, non_daf_donors]
  - label: label_green_revenue_next_365d
    cohorts: [all_donors]
    exclude_features: [some_column]   # optional
```

**Task type** is detected per label on the full training file: classification if the label has at most 10 distinct values, all whole numbers; regression otherwise. Classification labels must be 0/1.

**Mode** is decided per label (3.3), with nothing to set:

| | Validation | Scoring |
|---|---|---|
| When | the label window ends on or before today and the OOT labels are populated like the training labels | the window ends after today, or the OOT labels are (mostly) empty |
| OOT outputs | metrics with bootstrap intervals, deciles, `actual` column | scores and percentiles |

Model selection and the delivered scores are the same in both modes; the mode only changes what is evaluated and where outputs are filed.

**Outputs**: `Model Runs/<mode>/<label>/<cohort>/<run_date>` holds `model.pkl` (plain sklearn/LightGBM), `model_spec.json`, `metrics.json`, `predictions_oot.csv` and `explain.csv`. Each cohort folder is replaced in one step after all its files are written. `Model Runs/<mode>/<label>/run_summary.csv` records every cohort's status for the latest run. A PSI above 0.25 between training and OOT scores logs a warning. In validation mode, each cohort folder also gets report/.

**Config files** (all in the config folder):
- `modeling.yaml`: CV folds, feature selection, campaign depths and bootstrap settings, the candidate models and their grids, and model-selection overrides.
- `cohorts.yaml`: cohort definitions.
- `jobruns.yaml`: which labels run, on which cohorts.

Every file is checked on load: duplicate keys fail (plain YAML silently keeps the last of two same-named entries), and so do unknown keys in `modeling.yaml`, so typos and leftover options can't silently do nothing.

## 1. Run settings

In [ ]:
# Paths and inputs for this run; what to model lives in jobruns.yaml.
import os
from pathlib import Path

BASE_DIR   = Path('/Users/matt.fritz/Desktop/Donor Predictive Models')
CONFIG_DIR = BASE_DIR / 'Python and Config' / 'modeling_config'
DATA_DIR   = BASE_DIR / 'Constructed Data'
OUTPUT_DIR = BASE_DIR / 'Model Runs'
TRAIN_FILE = DATA_DIR / 'train_features_t2025oct19_oot2026oct19-2027oct19.csv'     # None = newest train_features_*.csv in DATA_DIR
OOT_FILE   = DATA_DIR / 'oot_features_t2025oct19_oot2026oct19-2027oct19.csv'       # None = the oot_features_*.csv paired with TRAIN_FILE

MIN_TRAIN_EVENTS = 30             # minimum training rows in the smaller class (non-zero rows for regression)
SEED = 42
LOG_LEVEL = 'INFO'

## 2. Imports and logging

In [ ]:
import hashlib
import json
import logging
import os
import pickle
import platform
import re
import shutil
import sys
import uuid
from dataclasses import dataclass
from datetime import date, datetime, timezone
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
import yaml
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge, TweedieRegressor
from sklearn.metrics import (average_precision_score, brier_score_loss, get_scorer, make_scorer,
                             mean_absolute_error, mean_squared_error, r2_score, roc_auc_score)
from sklearn.model_selection import KFold, ParameterGrid, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

if tuple(int(x) for x in re.findall(r'\d+', sklearn.__version__)[:2]) < (1, 4):
    raise ImportError(f'scikit-learn >= 1.4 is required, found {sklearn.__version__}')

# The library logs through 'donor_modeling'; third-party loggers stay at WARNING.
logging.basicConfig(stream=sys.stdout, level=logging.WARNING, force=True,
                    format='%(asctime)s %(levelname)-7s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('donor_modeling')
log.setLevel(LOG_LEVEL)
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

## 3. Library

No edits needed. Every function takes its settings as arguments (a `RunSettings` object) and reads no notebook globals, so these cells can move into a module unchanged.

### 3.1 Settings and configuration

In [ ]:
@dataclass(frozen=True)
class RunSettings:
    """Every knob the library reads, passed explicitly."""
    label: str
    task_type: str                              # 'classification' or 'regression'
    id_col: str = 'donor_id'
    non_feature_cols: tuple = ('donor_id', 'first_donation_date', 'last_donation_date')
    exclude_features: tuple = ()
    corr_threshold: float | None = None         # None = no correlation filter
    top_n_features: int | None = None           # None = no importance trimming
    cv_folds: int = 5
    cv_scoring: str | None = None               # sklearn scorer name or 'capture_at_<depth>'; None = task default
    manual_model: str | None = None
    min_train_events: int = 30
    depths: tuple = (0.01, 0.05, 0.10, 0.20)
    n_boot: int = 1000
    ci: float = 0.90
    psi_warn: float = 0.25                      # OOT-vs-training score PSI that logs a warning
    n_jobs: int = -1
    seed: int = 42

    @property
    def is_clf(self) -> bool:
        return self.task_type == 'classification'

    @property
    def scoring(self) -> str:
        return self.cv_scoring or ('roc_auc' if self.is_clf else 'neg_mean_squared_error')


def _check_keys(d, allowed, where):
    unknown = set(d or {}) - set(allowed)
    if unknown:
        raise ValueError(f'modeling.yaml {where}: unknown keys {sorted(unknown)}; allowed: {sorted(allowed)}')


def validate_modeling(m):
    """Structural checks on modeling.yaml, run once at startup: typos and leftover options fail before training."""
    _check_keys(m, {'cv_folds', 'feature_selection', 'evaluation', 'classification', 'regression', 'best_model_selection'}, 'top level')
    _check_keys(m.get('feature_selection'), {'correlation_threshold', 'top_n'}, 'feature_selection')
    _check_keys(m.get('evaluation'), {'depths', 'bootstrap'}, 'evaluation')
    _check_keys((m.get('evaluation') or {}).get('bootstrap'), {'n_boot', 'ci'}, 'evaluation.bootstrap')
    _check_keys(m.get('best_model_selection'), {'manual_selection', 'cv_scoring'}, 'best_model_selection')
    _check_keys((m.get('best_model_selection') or {}).get('cv_scoring'), {'classification', 'regression'}, 'best_model_selection.cv_scoring')
    depths = (m.get('evaluation') or {}).get('depths') or RunSettings.depths
    if not all(0 < d <= 1 for d in depths):
        raise ValueError(f'modeling.yaml evaluation.depths must be fractions in (0, 1], got {depths}')
    for task in ('classification', 'regression'):
        for name, c in (m.get(task) or {}).items():
            where = f'{task}.{name}'
            if (task, name) not in ESTIMATORS:
                raise ValueError(f'modeling.yaml {where}: unknown model; known: {[n for t, n in ESTIMATORS if t == task]}')
            _check_keys(c, {'enabled', 'base_params', 'cv_search', 'calibration'}, where)
            _check_keys(c.get('cv_search'), {'enabled', 'param_grid'}, f'{where}.cv_search')
            _check_keys(c.get('calibration'), {'enabled', 'method'}, f'{where}.calibration')
            if (c.get('cv_search') or {}).get('enabled') and not c['cv_search'].get('param_grid'):
                raise ValueError(f'modeling.yaml {where}: cv_search is enabled but has no param_grid')
    return m


def settings_from_config(modeling, *, label, task_type, exclude_features=(), min_train_events=30, seed=42):
    """RunSettings plus the enabled model configs for one label (modeling.yaml is validated at startup)."""
    model_cfgs = {n: c for n, c in (modeling.get(task_type) or {}).items() if c.get('enabled')}
    if not model_cfgs:
        raise ValueError(f'modeling.yaml enables no {task_type} models')
    fs, ev, sel = (modeling.get(k) or {} for k in ('feature_selection', 'evaluation', 'best_model_selection'))
    boot = ev.get('bootstrap') or {}
    settings = RunSettings(
        label=label, task_type=task_type,
        exclude_features=tuple(exclude_features),
        corr_threshold=fs.get('correlation_threshold'), top_n_features=fs.get('top_n'),
        cv_folds=modeling.get('cv_folds', 5),
        cv_scoring=(sel.get('cv_scoring') or {}).get(task_type),
        manual_model=sel.get('manual_selection'),
        min_train_events=min_train_events,
        depths=tuple(sorted(set(ev.get('depths') or RunSettings.depths))),
        n_boot=boot.get('n_boot', 1000), ci=boot.get('ci', 0.90),
        seed=seed,
    )
    return settings, model_cfgs


def to_jsonable(obj):
    """Recursively convert numpy/pandas values to JSON-safe builtins; NaN becomes null."""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, pd.DataFrame):
        return to_jsonable(obj.to_dict(orient='records'))
    if isinstance(obj, (list, tuple, set, np.ndarray, pd.Index)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, (bool, np.bool_)):
        return bool(obj)
    if isinstance(obj, (int, np.integer)):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, (datetime, pd.Timestamp)):
        return obj.isoformat()
    if isinstance(obj, Path):
        return str(obj)
    return obj


class _StrictLoader(yaml.SafeLoader):
    pass


def _mapping_without_duplicates(loader, node, deep=False):
    keys = [loader.construct_object(k, deep=deep) for k, _ in node.value]
    dupes = sorted({str(k) for k in keys if keys.count(k) > 1})
    if dupes:
        raise ValueError(f'duplicate keys {dupes} in the block starting at line {node.start_mark.line + 1}')
    return loader.construct_mapping(node, deep=deep)


_StrictLoader.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, _mapping_without_duplicates)


def load_yaml(path):
    """Parse a config file, rejecting duplicate keys (plain YAML silently keeps the last one)."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'missing config file {path}')
    try:
        return yaml.load(path.read_text(), Loader=_StrictLoader) or {}
    except ValueError as e:
        raise ValueError(f'{path.name}: {e}') from None


def write_json(obj, path):
    Path(path).write_text(json.dumps(to_jsonable(obj), indent=2))


def file_sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def environment():
    """Library versions the pickled model needs at scoring time."""
    return {'python': platform.python_version(), 'scikit-learn': sklearn.__version__,
            'lightgbm': lgb.__version__, 'numpy': np.__version__, 'pandas': pd.__version__}

### 3.2 Inputs, cohorts and job runs

In [ ]:
_FILE_DATE = r'(\d{4}[a-z]{3}\d{1,2})'   # e.g. 2026oct19 in the features notebook's file names


def _file_date(pattern, name):
    m = re.search(pattern, name)
    return datetime.strptime(m.group(1), '%Y%b%d').date().isoformat() if m else None


def resolve_input_files(data_dir, train_file=None, oot_file=None):
    """Explicit paths win. Otherwise the newest train_features_*.csv in data_dir (by the T date in its
    name, then modification time) and the oot_features_*.csv with the same suffix.
    """
    if train_file is None:
        found = sorted(Path(data_dir).glob('train_features_*.csv'),
                       key=lambda p: (_file_date(r'_t' + _FILE_DATE + '_', p.name.lower()) or '', p.stat().st_mtime))
        if not found:
            raise FileNotFoundError(f'no train_features_*.csv in {data_dir}')
        train_file = found[-1]
    train_file = Path(train_file)
    if oot_file is None:
        if not train_file.name.startswith('train_features_'):
            raise ValueError('set OOT_FILE: it can only be inferred from a train_features_*.csv name')
        oot_file = train_file.with_name('oot_features_' + train_file.name.removeprefix('train_features_'))
    oot_file = Path(oot_file)
    missing = [str(p) for p in (train_file, oot_file) if not p.exists()]
    if missing:
        raise FileNotFoundError(f'input files not found: {missing}')
    return train_file, oot_file


def normalize_id(ids):
    """Ids as clean strings, so joins can't silently miss on int/float/str mismatches ('123.0' -> '123')."""
    return ids.astype(str).str.strip().str.replace(r'\.0$', '', regex=True)


def read_features(path, id_col='donor_id'):
    """Read a feature CSV with unique normalized ids. Bool columns become 0/1: v1.2's
    numeric/categorical split matched neither, so any bool features were dropped.
    """
    df = pd.read_csv(path, low_memory=False)
    if id_col not in df.columns:
        raise KeyError(f'{Path(path).name}: no {id_col!r} column')
    df[id_col] = normalize_id(df[id_col])
    if df[id_col].duplicated().any():
        raise ValueError(f'{Path(path).name}: duplicate {id_col} values')
    bools = df.select_dtypes('bool').columns
    df[bools] = df[bools].astype('int8')
    return df


_OPS = {
    '==': lambda s, v: s == v,
    '!=': lambda s, v: s != v,               # keeps nulls, as v1.2 did
    '>': lambda s, v: s > v,
    '>=': lambda s, v: s >= v,
    '<': lambda s, v: s < v,
    '<=': lambda s, v: s <= v,
    'between': lambda s, v: s.between(v[0], v[1]),
    'not_null': lambda s, v: s.notna(),
    'is_null': lambda s, v: s.isna(),
}


def cohort_mask(df, name, cohorts):
    """Row mask for one cohort. Unknown cohorts, filter columns or operators raise instead of
    silently widening the cohort (v1.2 skipped them).
    """
    if name not in cohorts:
        raise KeyError(f'cohort {name!r} is not in cohorts.yaml')
    mask = pd.Series(True, index=df.index)
    for col, spec in (cohorts[name].get('filters') or {}).items():
        if col not in df.columns:
            raise KeyError(f'cohort {name!r}: filter column {col!r} is not in the data')
        if spec['operator'] not in _OPS:
            raise ValueError(f"cohort {name!r}: unknown operator {spec['operator']!r}")
        mask &= _OPS[spec['operator']](df[col], spec.get('value'))
    return mask


def load_jobruns(path, cohorts):
    """Runs from jobruns.yaml. Structural problems raise here, before any training starts."""
    path = Path(path)
    runs = load_yaml(path).get('runs') or []
    if not runs:
        raise ValueError(f'{path.name}: no runs listed')
    seen = set()
    for i, run in enumerate(runs, 1):
        where = f'{path.name} run {i}'
        if not isinstance(run, dict) or not run.get('label') or not isinstance(run.get('cohorts'), list) or not run['cohorts']:
            raise ValueError(f'{where}: needs a label and a non-empty cohorts list')
        unknown = set(run) - {'label', 'cohorts', 'exclude_features'}
        if unknown:
            raise ValueError(f'{where}: unknown keys {sorted(unknown)}')
        if run['label'] in seen:
            raise ValueError(f"{where}: {run['label']!r} is listed twice")
        seen.add(run['label'])
        missing = [c for c in run['cohorts'] if c not in cohorts]
        if missing or len(set(run['cohorts'])) < len(run['cohorts']):
            raise ValueError(f'{where}: unknown or repeated cohorts {missing or run["cohorts"]}')
        if not isinstance(run.get('exclude_features', []), list):
            raise ValueError(f'{where}: exclude_features must be a list')
    return runs


def split_xy(df, s):
    """Feature frame, label and ids. Drops ids, dates, every label_* column and excluded features."""
    if s.label not in df.columns:
        labels = sorted(c for c in df.columns if c.startswith('label_'))
        raise KeyError(f'{s.label!r} is not in the data; labels include {labels[:10]}')
    drop = set(s.non_feature_cols) | set(s.exclude_features)
    cols = [c for c in df.columns if c not in drop and not c.startswith('label_')]
    return df[cols], df[s.label], df[s.id_col]

### 3.3 Task type and mode

Task type uses the v1.2 rule, applied once per label to the full training file so every cohort of a label gets the same model type.

Only the OOT labels themselves show whether the label window has been observed: the file names give the window start (oot<start>) and the label name gives the horizon (_next_<N>d or _next_<N>m), but neither gives the extract date. Today’s date is an upper bound on the data, so:

- a window that ends after today can't have been observed → **scoring**;
- otherwise the OOT labels decide: populated at ≥ 50% of the training rate → **validation**; anything less → **scoring**, with a warning between 20% and 50% (a partly observed window).

Rates are the minority-class share for 0/1 labels (so near-100% labels such as `label_will_not_give_*` behave like near-0% ones) and the non-zero share for regression labels. If the file names carry no OOT date, or the label name has no horizon (e.g. label_time_to_next_gift_days), the labels decide alone.

In [ ]:
def detect_task_type(y):
    """v1.2 rule: classification if the label has at most 10 distinct values, all whole numbers;
    regression otherwise.
    """
    if not pd.api.types.is_numeric_dtype(y):
        raise ValueError(f'{y.name} is not numeric')
    values = y.dropna().unique()
    return 'classification' if len(values) <= 10 and np.all(np.mod(values, 1) == 0) else 'regression'


def resolve_mode(s, file_names, y_train, y_oot, today=None, populated=0.5, partial=0.2):
    """'validation' or 'scoring' for this label. Pass full (pre-cohort) label series: a small cohort
    can have no OOT events even when validation is possible.
    """
    m = re.search(r'_next_(\d+)([dm])$', s.label)      # horizon in days or months, if the label name has one
    starts = [d for d in (_file_date(r'_oot' + _FILE_DATE + '-', Path(n).name.lower()) for n in file_names) if d]
    window_end = None
    if m and starts:
        horizon = pd.Timedelta(days=int(m.group(1))) if m.group(2) == 'd' else pd.DateOffset(months=int(m.group(1)))
        window_end = pd.Timestamp(starts[0]) + horizon
    today = pd.Timestamp(today or date.today())

    def rate(y):
        y = y.dropna()
        if y.empty:
            return 0.0
        return min(y.mean(), 1 - y.mean()) if s.is_clf else y.ne(0).mean()

    if rate(y_train) == 0:
        raise ValueError(f'{s.label}: the training label has no events')
    ratio = rate(y_oot) / rate(y_train)
    if window_end is None:
        missing = 'horizon (_next_<N>d or _next_<N>m) in the label name' if m is None else 'OOT date in the file names'
        log.warning('%s: no %s; mode is decided from the labels alone', s.label, missing)
    if window_end is not None and window_end > today:
        mode = 'scoring'
        if ratio >= partial:
            log.warning('%s: OOT labels are at %.0f%% of the training rate although the window runs to %s; '
                        'they are ignored', s.label, 100 * ratio, window_end.date())
    elif ratio >= populated:
        mode = 'validation'
    else:
        mode = 'scoring'
        if ratio >= partial:
            log.warning('%s: OOT labels are only at %.0f%% of the training rate; treating the window as not '
                        'fully observed', s.label, 100 * ratio)
    log.info('%s: %s mode (label window ends %s; OOT label rate %.2f x training)', s.label, mode,
             'unknown' if window_end is None else window_end.date(), ratio)
    return mode, window_end

### 3.4 Feature selection and preprocessing

Feature selection is fit on each CV training fold (so CV scores stay honest) and once more on all training rows for the final model. It doesn't depend on the model, so each fold's selection is shared by every candidate and grid setting.

In [ ]:
def select_features(X, y, s):
    """Drop constant columns, then (optionally) one of each highly correlated pair, then keep the
    top-N columns by LightGBM gain. Returns column names in their original order.
    """
    cols = [c for c in X.columns if X[c].nunique(dropna=True) > 1]
    numeric = [c for c in cols if pd.api.types.is_numeric_dtype(X[c])]
    if s.corr_threshold is not None and len(numeric) > 1:
        # A 20k-row sample estimates correlations to about ±0.01 at a fraction of the cost.
        sample = X[numeric].sample(min(len(X), 20_000), random_state=s.seed)
        corr = sample.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
        dropped = {c for c in upper.columns if (upper[c] > s.corr_threshold).any()}
        cols = [c for c in cols if c not in dropped]
    if s.top_n_features is not None and len(cols) > s.top_n_features:
        codes = X[cols].apply(lambda c: c if pd.api.types.is_numeric_dtype(c)
                              else c.astype('category').cat.codes.replace(-1, np.nan))
        est = (lgb.LGBMClassifier if s.is_clf else lgb.LGBMRegressor)(
            n_estimators=100, learning_rate=0.1, num_leaves=15, importance_type='gain',
            random_state=s.seed, verbose=-1)
        est.fit(codes.to_numpy(dtype=float), y)
        keep = set(pd.Series(est.feature_importances_, index=cols).nlargest(s.top_n_features).index)
        cols = [c for c in cols if c in keep]
    return cols


def numeric_cols(X):
    return [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]


def make_preprocessor(family, numeric, categorical):
    if family == 'tree':
        # LightGBM handles missing values natively and needs no scaling; categories become codes.
        steps = [('num', 'passthrough', numeric),
                 ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan,
                                        encoded_missing_value=np.nan), categorical)]
    else:
        # Linear models: median-impute with missing indicators (so "unknown" stays a signal instead of
        # looking like the median), scale, and one-hot encode categories.
        steps = [('num', Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)),
                                   ('scale', StandardScaler())]), numeric),
                 ('cat', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                                   ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
                  categorical)]
    prep = ColumnTransformer(steps, remainder='drop')
    return prep.set_output(transform='pandas') if family == 'tree' else prep


def build_pipeline(est, family, X):
    """Plain sklearn pipeline (preprocessing + a copy of est) for the columns of X."""
    numeric = numeric_cols(X)
    prep = make_preprocessor(family, numeric, [c for c in X.columns if c not in numeric])
    return Pipeline([('prep', prep), ('model', clone(est))])

### 3.5 Models

Candidates are grid-searched over shared CV folds, and the winner is the highest CV score in both modes. `fit_final` refits it on all training rows, adding calibration only if `modeling.yaml` enables it. Errors inside CV raise rather than becoming silent NaN scores. Everything sent to worker processes is a plain sklearn object, so the code runs the same in a notebook and as a `.py` script.

In [ ]:
ESTIMATORS = {  # (task, name) -> (class, family)
    ('classification', 'logreg'): (LogisticRegression, 'linear'),
    ('classification', 'lgbm'): (lgb.LGBMClassifier, 'tree'),
    ('regression', 'ridge'): (Ridge, 'linear'),
    ('regression', 'tweedie'): (TweedieRegressor, 'linear'),
    ('regression', 'lgbm_regressor'): (lgb.LGBMRegressor, 'tree'),
}
DEFAULT_PARAMS = {'logreg': {'max_iter': 1000}, 'lgbm': {'verbose': -1}, 'lgbm_regressor': {'verbose': -1}}


def make_estimator(name, s, params=None):
    if (s.task_type, name) not in ESTIMATORS:
        raise KeyError(f'no {s.task_type} estimator called {name!r}; known: {[n for t, n in ESTIMATORS if t == s.task_type]}')
    cls, family = ESTIMATORS[(s.task_type, name)]
    params = {**DEFAULT_PARAMS.get(name, {}), **(params or {})}
    if 'random_state' in cls().get_params():
        params.setdefault('random_state', s.seed)
    return cls(**params), family


def make_cv(s):
    return (StratifiedKFold if s.is_clf else KFold)(n_splits=s.cv_folds, shuffle=True, random_state=s.seed)


def _capture_score(y_true, y_score, depth):
    return capture_at(np.asarray(y_true, float), np.asarray(y_score, float), [depth])[0]


def make_cv_scorer(s):
    """A named sklearn scorer, or 'capture_at_<depth>' to select on capture at a campaign depth."""
    m = re.fullmatch(r'capture_at_([0-9.]+)', s.scoring)
    if not m:
        return get_scorer(s.scoring)
    return make_scorer(_capture_score, depth=float(m.group(1)),
                       response_method='predict_proba' if s.is_clf else 'predict')


def _with_calibration(est, calibration, s):
    if s.is_clf and calibration and calibration.get('enabled'):
        return CalibratedClassifierCV(est, method=calibration.get('method', 'sigmoid'), cv=make_cv(s), ensemble=False)
    return est


def make_folds(X, y, s):
    """CV folds as (train rows, test rows, features selected on the train rows)."""
    splits = list(make_cv(s).split(X, y))
    selected = Parallel(n_jobs=s.n_jobs)(delayed(select_features)(X.iloc[tr], y.iloc[tr], s) for tr, _ in splits)
    return [(tr, te, feats) for (tr, te), feats in zip(splits, selected)]


def _fit_fold(est, family, X, y, tr, feats):
    return build_pipeline(est, family, X[feats]).fit(X.iloc[tr][feats], y.iloc[tr])


def _score_fold(est, family, X, y, tr, te, feats, scorer):
    return scorer(_fit_fold(est, family, X, y, tr, feats), X.iloc[te][feats], y.iloc[te])


def _predict_fold(est, family, X, y, tr, te, feats, is_clf):
    model, X_te = _fit_fold(est, family, X, y, tr, feats), X.iloc[te][feats]
    return model.predict_proba(X_te)[:, 1] if is_clf else model.predict(X_te)


def fit_candidate(name, X, y, s, model_cfg, folds):
    """Grid-search one model over the shared folds; returns its best CV score and parameters."""
    est, family = make_estimator(name, s, model_cfg.get('base_params'))
    search = model_cfg.get('cv_search') or {}
    grid = list(ParameterGrid((search.get('param_grid') or {}) if search.get('enabled') else {}))
    scorer = make_cv_scorer(s)
    scores = Parallel(n_jobs=s.n_jobs)(
        delayed(_score_fold)(clone(est).set_params(**p), family, X, y, tr, te, feats, scorer)
        for p in grid for tr, te, feats in folds)
    by_setting = np.asarray(scores, float).reshape(len(grid), len(folds))
    i = int(np.nanargmax(by_setting.mean(axis=1)))
    log.info('  %-15s CV %s %.4f ± %.4f over %d setting(s) %s', name, s.scoring, by_setting[i].mean(),
             by_setting[i].std(), len(grid), grid[i])
    return {'name': name, 'family': family, 'estimator': clone(est).set_params(**grid[i]),
            'cv_score': float(by_setting[i].mean()), 'cv_std': float(by_setting[i].std()), 'params': grid[i]}


def select_best(cands, s):
    """Manual override if configured, else the highest CV score (sklearn scorers are higher-is-better)."""
    if s.manual_model:
        if s.manual_model not in cands:
            raise ValueError(f'manual_selection {s.manual_model!r} was not trained')
        return s.manual_model
    scored = {m: c['cv_score'] for m, c in cands.items() if np.isfinite(c['cv_score'])}
    if not scored:
        raise RuntimeError('no candidate has a finite CV score')
    return max(scored, key=scored.get)


def fit_final(cand, X, y, s, calibration=None):
    """Refit a candidate on all training rows of X (already reduced to the final features)."""
    return _with_calibration(build_pipeline(cand['estimator'], cand['family'], X), calibration, s).fit(X, y)


def oof_scores(cand, X, y, s, folds):
    """Out-of-fold training scores before calibration, from the search's folds and selections. Ranking
    metrics don't need calibration, and uncalibrated scores have no isotonic ties, which keeps
    them usable as the drift reference.
    """
    parts = Parallel(n_jobs=s.n_jobs)(
        delayed(_predict_fold)(cand['estimator'], cand['family'], X, y, tr, te, feats, s.is_clf)
        for tr, te, feats in folds)
    oof = np.empty(len(y))
    for (_, te, _), p in zip(folds, parts):
        oof[te] = p
    return oof


def predict_scores(model, X, features, s):
    X = X[features]
    return model.predict_proba(X)[:, 1] if s.is_clf else model.predict(X)


def unwrap(model):
    """The prep+model pipeline inside an optional CalibratedClassifierCV."""
    return model.calibrated_classifiers_[0].estimator if isinstance(model, CalibratedClassifierCV) else model

### 3.6 Evaluation

All ranking goes through `rank_desc`, which breaks ties with a seeded random key, so tied scores never fall into bins by row order. Capture is the share of all events (or revenue) in the top slice; lift is capture divided by the slice's share of rows. A depth is `decision_grade` when the lower end of its lift interval is above 1.

In [ ]:
def rank_desc(scores, seed=0):
    """1 = highest score; ties broken by a seeded random key."""
    scores = np.asarray(scores, float)
    key = np.random.default_rng(seed).random(scores.size)
    order = np.lexsort((key, -scores))
    rank = np.empty(scores.size, dtype=np.int64)
    rank[order] = np.arange(1, scores.size + 1)
    return rank


def _top_k(depth, n):
    return max(1, int(np.ceil(depth * n)))


def capture_at(y, scores, depths, seed=0):
    """Share of all events (or revenue) in the top `depth` of scores, for each depth."""
    rank, total = rank_desc(scores, seed), y.sum()
    return np.array([y[rank <= _top_k(d, y.size)].sum() / total if total else np.nan for d in depths])


def bootstrap_ci(stat, *arrays, strata=None, n_boot=1000, ci=0.90, seed=0):
    """stat(*arrays) with a percentile interval from resampling rows (within strata if given).
    Pass several score arrays for a paired comparison. n_boot=0 returns no interval.
    """
    point = np.asarray(stat(*arrays), float)
    if n_boot <= 0:
        return point, np.full_like(point, np.nan), np.full_like(point, np.nan)
    rng = np.random.default_rng(seed)
    groups = [np.flatnonzero(strata == v) for v in np.unique(strata)] if strata is not None else [np.arange(len(arrays[0]))]
    draws = []
    for _ in range(n_boot):
        # Shuffle after stratified sampling so tie-breaking can't line up with the strata.
        idx = rng.permutation(np.concatenate([rng.choice(g, size=g.size) for g in groups]))
        draws.append(stat(*(a[idx] for a in arrays)))
    lo, hi = np.nanpercentile(np.asarray(draws, float), [50 * (1 - ci), 50 * (1 + ci)], axis=0)
    return point, lo, hi


def depth_table(y, scores, s, n_boot=0):
    """Capture and lift at each campaign depth, with intervals and the share of the slice decided by ties."""
    n, depths = y.size, list(s.depths)
    cap, lo, hi = bootstrap_ci(lambda yy, ss: capture_at(yy, ss, depths, s.seed), y, scores,
                               strata=y if s.is_clf else None, n_boot=n_boot, ci=s.ci, seed=s.seed)
    k = np.array([_top_k(d, n) for d in depths])
    ordered = np.sort(scores)[::-1]
    tie_share = []
    for kk in k:  # rows in the slice whose membership depends on tie-breaking
        in_slice, in_all = (ordered[:kk] == ordered[kk - 1]).sum(), (scores == ordered[kk - 1]).sum()
        tie_share.append(in_slice / kk if in_all > in_slice else 0.0)
    share = k / n
    t = pd.DataFrame({'depth': depths, 'n_top': k, 'capture': cap, 'capture_lo': lo, 'capture_hi': hi,
                      'lift': cap / share, 'lift_lo': lo / share, 'lift_hi': hi / share,
                      'tie_broken_share': tie_share})
    t['decision_grade'] = (t['lift_lo'] > 1).where(t['lift_lo'].notna())
    return t


def decile_table(y, scores, s, n_bins=10):
    """Equal-size score bins, bin 1 = highest scores, with cumulative capture and lift."""
    b = np.ceil(rank_desc(scores, s.seed) * n_bins / y.size).astype(int)
    t = (pd.DataFrame({'bin': b, 'y': y, 'score': scores}).groupby('bin')
         .agg(n=('y', 'size'), total=('y', 'sum'), mean=('y', 'mean'),
              min_score=('score', 'min'), max_score=('score', 'max')).reset_index())
    t['lift'] = t['mean'] / y.mean()
    t['cum_capture'] = t['total'].cumsum() / y.sum()
    t['cum_lift'] = t['cum_capture'] / (t['n'].cumsum() / y.size)
    return t.rename(columns={'total': 'events', 'mean': 'event_rate'}) if s.is_clf else t


def normalized_gini(y, scores, seed=0):
    def gini(order_by):
        cum = np.cumsum(y[np.argsort(rank_desc(order_by, seed))]) / y.sum()
        return cum.mean() - (y.size + 1) / (2 * y.size)
    return gini(scores) / gini(y)


def evaluate(y, scores, s, n_boot=0, calibrated=False):
    """Headline metrics plus depth and decile tables. Point estimates only when n_boot=0."""
    y, scores = np.asarray(y, float), np.asarray(scores, float)
    m = {'n': y.size, 'n_unique_scores': np.unique(scores).size}
    if s.is_clf:
        m.update(events=y.sum(), base_rate=y.mean())
        if 0 < y.sum() < y.size:
            m.update(auc=roc_auc_score(y, scores), pr_auc=average_precision_score(y, scores))
            if calibrated:  # only meaningful when the probability itself is used
                m['brier_skill'] = 1 - brier_score_loss(y, scores) / brier_score_loss(y, np.full(y.size, y.mean()))
    else:
        m.update(rmse=np.sqrt(mean_squared_error(y, scores)), mae=mean_absolute_error(y, scores),
                 r2=r2_score(y, scores) if y.size > 1 else np.nan,
                 baseline_rmse=np.sqrt(mean_squared_error(y, np.full(y.size, y.mean()))),
                 gini=normalized_gini(y, scores, s.seed) if y.sum() else np.nan)
    m['depths'] = depth_table(y, scores, s, n_boot)
    m['deciles'] = decile_table(y, scores, s)
    return m


def psi_bins(reference, bins=10):
    """Decile edges of the reference scores and the reference share in each bin."""
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    return edges, _bin_shares(reference, edges)


def _bin_shares(x, edges):
    counts = np.histogram(np.clip(x, edges[0], edges[-1]), edges)[0]
    return np.clip(counts / len(x), 1e-6, None)


def psi(reference, current, bins=10):
    """Population stability index of current scores against the reference distribution."""
    edges, ref = psi_bins(reference, bins)
    if edges.size < 3:
        return np.nan
    cur = _bin_shares(current, edges)
    return float(np.sum((cur - ref) * np.log(cur / ref)))

### 3.7 Explanation

`explain.csv` holds coefficients for linear winners and LightGBM gain for tree winners.

In [ ]:
def explain(model, family, s):
    """Linear: coefficients (numeric and missing-indicator inputs are standardized, so those are per SD;
    one-hot terms are per category). Trees: LightGBM gain.
    """
    pipe = unwrap(model)
    prep, est = pipe.named_steps['prep'], pipe.named_steps['model']
    names = [re.sub(r'^(num|cat)__', '', n) for n in prep.get_feature_names_out()]
    if family == 'linear':
        coef = np.ravel(est.coef_)
        table = pd.DataFrame({'feature': names, 'coef': coef})
        if s.is_clf:
            table['odds_ratio'] = np.exp(coef)
        return table.iloc[np.argsort(-np.abs(coef))].reset_index(drop=True)
    gain = est.booster_.feature_importance('gain')
    return pd.DataFrame({'feature': names, 'gain': gain}).sort_values('gain', ascending=False, ignore_index=True)

### 3.8 Runners and publishing

- `train_cohort` raises `CohortSkipped` when a cohort can't be modeled this run: nothing to score, or too few rows in the smaller class for stratified CV (for a 99.7%-positive label, that's the negatives). It logs a warning when OOT scores shift from training scores by more than PSI 0.25.
- `publish_outputs` writes everything to a staging folder and swaps it in only after every file is written, so a cohort folder always holds one complete run. `model_version` is a hash of the pickled model.
- `run_label` runs one `jobruns.yaml` entry: it checks the label, detects its task type and mode, then trains and publishes each cohort. It returns one status row per cohort and never raises for a single label or cohort.

In [ ]:
class CohortSkipped(Exception):
    """A cohort that can't be modeled this run; logged, not treated as a failure."""
    pass


def train_cohort(name, train, oot, mode, s, model_cfgs):
    """Tune, select, refit, evaluate and explain one cohort."""
    X, y, _ = split_xy(train, s)
    X_oot, y_oot, ids_oot = split_xy(oot, s)
    if X_oot.empty:
        raise CohortSkipped('no OOT rows to score')
    if y.isna().any():  # e.g. rate labels that are undefined for some donors
        log.info('%s / %s: dropping %d training rows with no label', s.label, name, y.isna().sum())
        X, y = X[y.notna()], y[y.notna()]
    # Stratified CV needs enough of the smaller class in every fold (non-zero rows for regression).
    events = int((y != 0).sum())
    minority = min(events, len(y) - events) if s.is_clf else events
    need = max(s.min_train_events, s.cv_folds)
    if minority < need:
        what = 'rows in the smaller class' if s.is_clf else 'non-zero rows'
        raise CohortSkipped(f'{minority} training {what}, need {need}')
    log.info('%s / %s: %s training rows, %s events; %s OOT rows', s.label, name, f'{len(y):,}', f'{events:,}', f'{len(X_oot):,}')

    folds = make_folds(X, y, s)
    cands = {}
    for m, cfg in model_cfgs.items():
        try:
            cands[m] = fit_candidate(m, X, y, s, cfg, folds)
        except Exception:
            log.exception('%s / %s: %s failed', s.label, name, m)
    if not cands:
        raise RuntimeError('every model failed')
    best = select_best(cands, s)
    calibration = model_cfgs[best].get('calibration')
    features = select_features(X, y, s)                           # final selection on all training rows
    model = fit_final(cands[best], X[features], y, s, calibration)
    oof = oof_scores(cands[best], X, y, s, folds)
    scores = predict_scores(model, X_oot, features, s)            # delivered scores (calibrated if configured)
    calibrated = isinstance(model, CalibratedClassifierCV)
    raw_oot = predict_scores(unwrap(model), X_oot, features, s) if calibrated else scores
    r = {'cohort': name, 'best': best, 'family': cands[best]['family'], 'model': model,
         'features': features, 'calibration': model.method if calibrated else None,
         'candidates': pd.DataFrame([{k: c[k] for k in ('name', 'cv_score', 'cv_std', 'params')}
                                     for c in cands.values()]).rename(columns={'name': 'model'}),
         'n_train': len(y), 'train_events': events, 'oof': oof,
         'oot_ids': ids_oot.to_numpy(), 'oot_scores': scores,
         'metrics': {'train_oof': evaluate(y, oof, s)},
         'psi_oot_vs_oof': psi(oof, raw_oot)}
    log.info('  best: %s on %d features; OOT score PSI vs training %.3f', best, len(features), r['psi_oot_vs_oof'])
    if r['psi_oot_vs_oof'] > s.psi_warn:
        log.warning('%s / %s: OOT scores shifted from training scores (PSI %.3f > %.2f); check the OOT extract',
                    s.label, name, r['psi_oot_vs_oof'], s.psi_warn)

    if mode == 'validation':
        ok = y_oot.notna().to_numpy()
        r['y_oot'] = y_oot.to_numpy(float)
        y_ok = r['y_oot'][ok]
        if not ok.any():
            log.warning('  no labeled OOT rows; OOT evaluation skipped')
        else:
            r['metrics']['oot'] = evaluate(y_ok, scores[ok], s, n_boot=s.n_boot, calibrated=calibrated)
    r['explain'] = explain(model, r['family'], s)
    return r


def model_spec(r, s, mode, window_end, inputs, version):
    """Everything needed to reproduce and audit this model."""
    params = r['candidates'].set_index('model').loc[r['best'], 'params']
    oot = r['metrics'].get('oot')
    headline = ('n', 'events', 'base_rate', 'auc', 'pr_auc', 'brier_skill', 'rmse', 'mae', 'r2', 'gini')
    return {
        'model_version': f"{s.label}__{r['cohort']}__{version}",
        'label': s.label, 'task_type': s.task_type, 'mode': mode,
        'created_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'training_population': r['cohort'],
        'feature_list': r['features'],
        'model': {'name': r['best'], 'family': r['family'], 'params': params, 'calibration': r['calibration']},
        'selection': {'cv_scoring': s.scoring, 'cv_folds': s.cv_folds, 'candidates': r['candidates']},
        'inputs': inputs,
        'training': {'rows': r['n_train'], 'events': r['train_events'],
                     'label_window_end': None if window_end is None else window_end.date().isoformat()},
        'validation': None if oot is None else {**{k: oot[k] for k in headline if k in oot}, 'depths': oot['depths']},
        'environment': environment(),
    }


def publish_outputs(r, s, mode, window_end, inputs, out_dir):
    """Stage every file next to out_dir, then swap the folder in, so out_dir always holds one complete run."""
    out_dir = Path(out_dir)
    out_dir.parent.mkdir(parents=True, exist_ok=True)
    for leftover in out_dir.parent.glob(f'.{out_dir.name}.*-*'):   # from an interrupted earlier run
        shutil.rmtree(leftover, ignore_errors=True)
    blob = pickle.dumps(r['model'])
    spec = model_spec(r, s, mode, window_end, inputs, hashlib.sha256(blob).hexdigest()[:12])
    tag = uuid.uuid4().hex[:8]
    stage, old = out_dir.parent / f'.{out_dir.name}.new-{tag}', out_dir.parent / f'.{out_dir.name}.old-{tag}'
    stage.mkdir()
    try:
        (stage / 'model.pkl').write_bytes(blob)
        write_json(spec, stage / 'model_spec.json')
        write_json({'candidates': r['candidates'], **r['metrics'], 'psi_oot_vs_oof': r['psi_oot_vs_oof']},
                   stage / 'metrics.json')
        preds = pd.DataFrame({'model_name': s.label, 'cohort': r['cohort'], 'score_date': out_dir.name, s.id_col: r['oot_ids'], 'score': r['oot_scores'],
                              'percentile': 100 * pd.Series(r['oot_scores']).rank(pct=True).to_numpy()})
        if mode == 'validation':
            preds['actual'] = r['y_oot']
        preds.to_csv(stage / 'predictions_oot.csv', index=False)
        r['explain'].to_csv(stage / 'explain.csv', index=False)
        if out_dir.exists():
            out_dir.rename(old)
        stage.rename(out_dir)
    except BaseException:
        shutil.rmtree(stage, ignore_errors=True)
        if old.exists() and not out_dir.exists():
            old.rename(out_dir)   # put the previous run back
        raise
    shutil.rmtree(old, ignore_errors=True)
    log.info('  published %s (%s)', out_dir, spec['model_version'])
    return spec


def cohort_summary(r, s):
    """Run-summary columns for a published cohort; validation runs add OOT results at each depth."""
    row = {'best': r['best'], 'features': len(r['features']), 'train_rows': r['n_train'],
           'train_events': r['train_events'], 'psi_oot_vs_oof': r['psi_oot_vs_oof'],
           'cv_score': r['candidates'].set_index('model').loc[r['best'], 'cv_score']}
    oot = r['metrics'].get('oot')
    if oot is not None:
        row['oot_auc' if s.is_clf else 'oot_gini'] = oot.get('auc', oot.get('gini'))
        for d in oot['depths'].itertuples():
            row[f'lift_{d.depth:g}'], row[f'lift_lo_{d.depth:g}'] = d.lift, d.lift_lo
    return row


def run_label(run, train_df, oot_df, modeling, cohorts, inputs, file_names, output_dir, min_train_events=30, seed=42):
    """Train and publish every cohort for one jobruns.yaml entry; returns one status row per cohort."""
    label, run_date = run['label'], datetime.now(timezone.utc).date().isoformat()
    try:
        missing = [c for c in [label, *run.get('exclude_features', [])] if c not in train_df.columns or c not in oot_df.columns]
        if missing:
            raise KeyError(f'not in both input files: {missing}')
        task = detect_task_type(train_df[label])
        if task == 'classification' and not set(train_df[label].dropna().unique()) <= {0, 1}:
            raise ValueError(f'{label} has at most 10 whole-number values, so it is classification, but it is not 0/1; '
                             'only 0/1 classification labels are supported')
        s, model_cfgs = settings_from_config(modeling, label=label, task_type=task, exclude_features=run.get('exclude_features', []),
                                             min_train_events=min_train_events, seed=seed)
        mode, window_end = resolve_mode(s, file_names, train_df[label], oot_df[label])
    except Exception as e:
        log.exception('%s: failed before training', label)
        return pd.DataFrame([{'label': label, 'cohort': None, 'status': 'failed', 'detail': f'{type(e).__name__}: {e}'}])

    log.info('%s: %s, %s mode; models %s; cohorts %s', label, task, mode, list(model_cfgs), run['cohorts'])
    rows = []
    for name in run['cohorts']:
        row = {'label': label, 'task_type': task, 'mode': mode, 'cohort': name}
        try:
            r = train_cohort(name, train_df[cohort_mask(train_df, name, cohorts)],
                             oot_df[cohort_mask(oot_df, name, cohorts)], mode, s, model_cfgs)
            spec = publish_outputs(r, s, mode, window_end, inputs, output_dir / mode / label / name / run_date)
            row.update(status='published', model_version=spec['model_version'], detail='', **cohort_summary(r, s))
        except CohortSkipped as e:
            log.warning('%s / %s: skipped, %s', label, name, e)
            row.update(status='skipped', detail=str(e))
        except Exception as e:
            log.exception('%s / %s: failed', label, name)
            row.update(status='failed', detail=f'{type(e).__name__}: {e}')
        rows.append(row)

    summary = pd.DataFrame(rows)
    out_root = output_dir / mode / label
    out_root.mkdir(parents=True, exist_ok=True)
    tmp = out_root / f'.run_summary.{uuid.uuid4().hex[:8]}.csv'
    summary.assign(run_at=datetime.now(timezone.utc).isoformat(timespec='seconds'), **inputs).to_csv(tmp, index=False)
    os.replace(tmp, out_root / f'run_summary_{run_date}.csv')
    return summary

## 4. Load configuration and inputs

Configuration problems (duplicate or unknown keys, unknown models or cohorts, a malformed `jobruns.yaml`) stop the run here, before any training.

In [ ]:
modeling = validate_modeling(load_yaml(CONFIG_DIR / 'modeling.yaml'))
cohorts = load_yaml(CONFIG_DIR / 'cohorts.yaml').get('cohorts') or {}
jobruns = load_jobruns(CONFIG_DIR / 'jobruns.yaml', cohorts)
log.info('jobruns.yaml: %d label(s): %s', len(jobruns), [r['label'] for r in jobruns])

In [ ]:
train_path, oot_path = resolve_input_files(DATA_DIR, TRAIN_FILE, OOT_FILE)
train_df = read_features(train_path)
oot_df = read_features(oot_path)
inputs = {'train_file': train_path.name, 'train_sha256': file_sha256(train_path),
          'oot_file': oot_path.name, 'oot_sha256': file_sha256(oot_path),
          't_date': _file_date(r'_t' + _FILE_DATE + '_', train_path.name.lower()),
          'oot_start': _file_date(r'_oot' + _FILE_DATE + '-', oot_path.name.lower())}
log.info('Train %s %s; OOT %s %s', train_path.name, train_df.shape, oot_path.name, oot_df.shape)

wide = {c: train_df[c].nunique() for c in train_df.columns
        if not c.startswith('label_') and c not in RunSettings.non_feature_cols
        and not pd.api.types.is_numeric_dtype(train_df[c]) and train_df[c].nunique() > 50}
if wide:
    log.warning('High-cardinality categoricals (one-hot encoded for linear models): %s', wide)

## 5. Train and publish

Each label, and each cohort within it, is trained and published independently. A skipped or failed cohort keeps its previous outputs and is recorded in that label's `run_summary.csv`.

In [ ]:
summaries = [run_label(run, train_df, oot_df, modeling, cohorts, inputs, [oot_path.name, train_path.name],
                       OUTPUT_DIR, MIN_TRAIN_EVENTS, SEED)
             for run in jobruns]

## 6. Validation report

Runs only for cohorts published in **validation** mode, where the holdout labels have been observed; in scoring mode it prints one line and does nothing else. It reads each cohort's published files plus the in-memory training and holdout frames, works for every model and for classification and regression labels, and saves its tables (CSV) and charts (PNG) to `<cohort>/report/`.

| Output | Answers |
|---|---|
| Validation scorecard | Did performance hold up out of time? Did the outcome rate or score distribution shift? |
| Targeting performance | If we select the top X%, how much of the outcome do we reach, versus random and versus the benchmark? |
| Outcome capture by audience size | The same comparison as a curve |
| Observed outcome by score decile | Do higher-priority scores mean more of the outcome? |
| Performance by donor segment | Does the model rank well within the largest distinct cohorts in `cohorts.yaml`? (all of them in the CSV) |
| Characteristics of selected donors | How does the top 10% differ from everyone else? |
| Factors associated with model scores | The 10 strongest features, and which way they lean |

Ranking quality is AUC for 0/1 labels and Spearman correlation for regression labels. Capture and lift views appear only for 0/1 labels and nonnegative, additive regression outcomes; rates, ratios, time-to-event and signed labels get the decile chart and ranking and error measures instead. Labels listed in `PRIORITY` as `'lower'` (e.g. time to next gift) rank the lowest predictions first. Revenue-type labels (`CURRENCY_PATTERN`) are shown in dollars.

In [ ]:
# Validation report. Runs only for cohorts published in validation mode (holdout labels observed);
# in scoring mode it prints one line and does nothing else.
BASELINE_RULES = {}           # per label, e.g. {'label_will_start_monthly_next_365d': {'feature': 'days_since_last_gift', 'higher_first': False}};
                              # labels not listed are compared with the single strongest model feature in training
PRIORITY = {'label_time_to_next_gift_days': 'lower'}
                              # labels where lower predictions are the priority; every other label is 'higher'
MIN_SEGMENT_EVENTS = 20       # segments with fewer outcome-positive donors (donors with nonzero outcomes, or donors
                              # for non-additive regression labels) are too small to judge
MAX_SEGMENTS = 10             # segments displayed; segment_performance.csv keeps all of them
NON_ADDITIVE_PATTERN = r'rate|ratio|pct|share|time_to'
                              # regression labels whose totals aren't meaningful targets (rates, ratios, time-to-event):
                              # no capture/lift views. Labels with negative values are treated the same way.
CURRENCY_PATTERN = r'revenue|amt|amount|dollar|max_single_gift'
                              # regression labels shown in dollars

from scipy.stats import rankdata

try:
    from IPython.display import display

    def _show(df):
        display(df.style.hide(axis='index'))
except ImportError:
    def _show(df):
        print(df.to_string(index=False))

_ACRONYMS = {'daf': 'DAF', 'cv': 'CV', 'eoy': 'EOY', 'mlg': 'MLG', 'trd': 'TRD', 'pct': '%'}


def _pretty(name):
    """Readable form of a snake_case label or feature name for display; saved files keep the raw names."""
    raw = name.removeprefix('label_')
    missing = raw.startswith('missingindicator_')
    tokens = raw.removeprefix('missingindicator_').split('_')
    words = []
    for i, t in enumerate(tokens):
        if re.fullmatch(r'\d+m', t):
            t = f'{t[:-1]} months'
        elif re.fullmatch(r'\d+d', t):
            t = f'{t[:-1]} days'
        elif t == 'next' and i + 1 < len(tokens) and re.fullmatch(r'\d+[dm]', tokens[i + 1]):
            t = 'in the next'
        words.append(_ACRONYMS.get(t.lower(), t))
    text = ' '.join(words)
    text = text[:1].upper() + text[1:]
    return f'{text} is missing' if missing else text


def _cohort_name(name):
    return (cohorts.get(name) or {}).get('description') or _pretty(name)


def _num(v, money=False):
    """Readable value: $30.0M, $2.82M, $5,362, 14.71, 0.0999 (no $ unless money)."""
    if v is None or not np.isfinite(v):
        return '—'
    sign, a, cur = ('-' if v < 0 else ''), abs(v), ('$' if money else '')
    if a >= 1e9:
        return f'{sign}{cur}{a / 1e9:.{2 if a < 1e10 else 1}f}B'
    if a >= 1e6:
        return f'{sign}{cur}{a / 1e6:.{2 if a < 1e7 else 1 if a < 1e8 else 0}f}M'
    if a >= 1000:
        return f'{sign}{cur}{a:,.0f}'
    return f'{sign}{cur}{a:,.2f}' if a >= 1 else f'{sign}{cur}{a:.3g}'


def _pct(v, digits=1):
    return '—' if v is None or not np.isfinite(v) else f'{100 * v:.{digits}f}%'


def _spearman(y, p):
    return float(np.corrcoef(rankdata(y), rankdata(p))[0, 1])


def _metric_view(name, value):
    """Display name, display value and whether higher is better, for a CV scorer name."""
    if name == 'neg_mean_squared_error':
        return 'RMSE', float(np.sqrt(-value)), False
    if name.startswith('neg_'):
        return name[4:].replace('_', ' '), -value, False
    return {'roc_auc': 'AUC', 'average_precision': 'PR-AUC'}.get(name, name.replace('_', ' ')), value, True


def _feature_values(df, name):
    """Raw values behind an explain.csv feature name (missing indicator -> isna, one-hot term -> match), or None."""
    raw = name.removeprefix('missingindicator_')
    if raw != name and raw in df.columns:
        return df[raw].isna().astype(float)
    if name in df.columns:
        return df[name].astype(float) if pd.api.types.is_numeric_dtype(df[name]) else None
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]) and name.startswith(c + '_'):
            return (df[c].astype(object).where(df[c].notna(), 'missing').astype(str) == name[len(c) + 1:]).astype(float)
    return None


def _strongest_feature(train_c, s, features):
    """The model feature whose rank correlation with the label in training is largest, with its direction."""
    X = train_c[[f for f in features if pd.api.types.is_numeric_dtype(train_c[f])]]
    keep = train_c[s.label].notna()
    rho = X[keep].fillna(X.median()).corrwith(train_c.loc[keep, s.label], method='spearman').dropna()
    best = rho.abs().idxmax()
    return {'feature': best, 'higher_first': bool(rho[best] > 0)}


def _save_show(fig, path):
    import matplotlib.pyplot as plt
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()


def _outcome_noun(label):
    """'label_project_revenue_next_365d' -> 'project revenue'."""
    return _pretty(re.sub(r'_next_\d+[dm]$', '', label)).lower()


def validation_report(label, cohort, s, run_dir, train_c, oot_c, rule=None, priority='higher'):
    """Tables and charts for one cohort published in validation mode."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    from matplotlib.ticker import PercentFormatter

    spec = json.loads((run_dir / 'model_spec.json').read_text())
    metrics = json.loads((run_dir / 'metrics.json').read_text())
    if 'oot' not in metrics:
        print(f'{label} / {cohort}: no labeled holdout rows; nothing to report')
        return
    preds = pd.read_csv(run_dir / 'predictions_oot.csv', dtype={s.id_col: str}).dropna(subset=['actual'])
    hold = oot_c.set_index(s.id_col).loc[preds[s.id_col]]          # same rows, same order as the predictions
    y, score, n = preds['actual'].to_numpy(float), preds['score'].to_numpy(float), len(preds)
    train_y = train_c[s.label].dropna().to_numpy(float)

    lower = priority == 'lower'
    prio = -score if lower else score                              # ordering used to select donors
    prio_text = 'lowest predictions first' if lower else 'highest scores first'
    # Capture and lift need an outcome whose total is meaningful and where more of it among selected donors is the goal.
    targeting = not lower and (s.is_clf or (min(y.min(), train_y.min()) >= 0 and not re.search(NON_ADDITIVE_PATTERN, label)))
    money = not s.is_clf and bool(re.search(CURRENCY_PATTERN, label))
    rank_name, rank_fn, rank_null = ('AUC', roc_auc_score, 0.5) if s.is_clf else ('Spearman correlation', _spearman, 0.0)
    outcome_word = 'Outcome rate' if s.is_clf else 'Average outcome'
    positive_word = 'outcome-positive donors' if s.is_clf else 'donors with nonzero outcomes' if targeting else None
    fmt_outcome = (lambda v: _pct(v, 2)) if s.is_clf else (lambda v: _num(v, money))
    strata, depths, boot = (y if s.is_clf else None), list(s.depths), dict(n_boot=s.n_boot, ci=s.ci, seed=s.seed)
    focus = min(depths, key=lambda v: abs(v - 0.10))
    ci = f'{s.ci:.0%} CI'
    title = f'{_pretty(label)} — {_cohort_name(cohort)}'
    out = run_dir / 'report'
    shutil.rmtree(out, ignore_errors=True)
    out.mkdir()

    rule = rule or _strongest_feature(train_c, s, spec['feature_list'])
    x = hold[rule['feature']].astype(float).fillna(train_c[rule['feature']].median())
    rule_score = (x if rule['higher_first'] else -x).to_numpy()   # oriented to rise with the outcome
    rule_text = f"{_pretty(rule['feature'])}, {'highest' if rule['higher_first'] else 'lowest'} first"

    counts = lambda v: f", {int((v != 0).sum()):,} {positive_word}" if positive_word else ''
    print(f"\n{'=' * 100}\n{title}\n"
          f"Model: {spec['model']['name']} on {len(spec['feature_list'])} features (version {spec['model_version'].rsplit('__', 1)[-1]})"
          f"{'; priority: lowest predictions' if lower else ''}\n"
          f"Training: {len(train_y):,} donors{counts(train_y)}  |  Holdout: {n:,} donors{counts(y)}\n{'=' * 100}")

    # Validation scorecard
    rows = []
    if targeting or s.is_clf:    # nonnegative outcomes: compare as a ratio
        ratio = y.mean() / train_y.mean()
        shift = f'Holdout is {ratio:.2f}× training' + (' (similar)' if 0.9 <= ratio <= 1.1 else '')
    else:                        # rates, ratios and signed outcomes: compare as a difference
        shift = f'Holdout minus training: {_num(y.mean() - train_y.mean(), money)}'
    rows.append((f'{outcome_word}, training → holdout', f'{fmt_outcome(train_y.mean())} → {fmt_outcome(y.mean())}', shift))
    best = next(c for c in spec['selection']['candidates'] if c['model'] == spec['model']['name'])
    model = pickle.loads((run_dir / 'model.pkl').read_bytes())
    cv_name, cv_val, higher_better = _metric_view(s.scoring, best['cv_score'])
    _, hold_val, _ = _metric_view(s.scoring, make_cv_scorer(s)(model, hold[spec['feature_list']], preds['actual']))
    rq, rq_lo, rq_hi = bootstrap_ci(rank_fn, y, score, strata=strata, **boot)
    if s.is_clf and s.scoring == 'roc_auc':     # the selection metric is the ranking measure: one row
        if cv_val > rq_hi:
            note = 'Declined over time: CV value is above the holdout interval'
        elif cv_val < rq_lo:
            note = 'Holdout higher than CV: CV value is below the holdout interval'
        else:
            note = 'No evidence of decline: CV value is inside the holdout interval'
        rows.append(('AUC, CV → holdout', f'{cv_val:.3f} → {rq:.3f} ({ci} {rq_lo:.3f}–{rq_hi:.3f})', note))
    else:
        in_units = cv_name in ('RMSE', 'mean absolute error')
        show = (lambda v: _num(v, money)) if in_units else (lambda v: f'{v:.4g}')
        change = (hold_val - cv_val) / abs(cv_val) if cv_val else np.nan
        noun = 'error' if not higher_better else cv_name
        rows.append((f'{cv_name}, CV → holdout', f'{show(cv_val)} → {show(hold_val)}',
                     f"Holdout {noun} is {abs(change):.1%} {'higher' if change >= 0 else 'lower'} than CV"))
        rows.append((f'{rank_name}, holdout', f'{rq:.3f} ({ci} {rq_lo:.3f}–{rq_hi:.3f})',
                     f'Scores rank outcomes better than chance (interval excludes {rank_null:g})' if rq_lo > rank_null
                     else f'No evidence that scores rank outcomes (interval includes {rank_null:g})'))

    total = y.sum()
    if targeting:
        d = depth_table(y, prio, s, n_boot=s.n_boot)
        stat = lambda yy, a, b: capture_at(yy, a, depths, s.seed) - capture_at(yy, b, depths, s.seed)
    else:
        stat = lambda yy, a, b: np.array([_spearman(yy, a) - _spearman(yy, b)])
    diff, dlo, dhi = bootstrap_ci(stat, y, prio if targeting else score, rule_score, strata=strata, **boot)
    if targeting:
        f = d.loc[(d['depth'] - focus).abs().idxmin()]
        rows.append((f'Top {focus:.0%} capture', f"{_pct(f['capture'])} ({f['lift']:.2f}× random)",
                     f"Better than random: lift {ci} {f['lift_lo']:.2f}–{f['lift_hi']:.2f}×" if f['lift_lo'] > 1
                     else 'Not distinguishable from random (lift interval includes 1×)'))
        k = depths.index(focus)
        extra = diff[k] * total
        more = 'more' if extra >= 0 else ('fewer' if s.is_clf else 'less')
        gain = (f'{abs(extra):,.0f} {more} outcome-positive donors among those selected' if s.is_clf
                else f'{_num(abs(extra), money)} {more} observed {_outcome_noun(label)} among selected donors')
        rows.append((f'Advantage vs single-feature benchmark, top {focus:.0%}',
                     f'{100 * diff[k]:+.2f} pts ({ci} {100 * dlo[k]:+.2f} to {100 * dhi[k]:+.2f})',
                     gain + ('' if dlo[k] > 0 or dhi[k] < 0 else '; interval includes 0, so no clear advantage')))
    else:
        rows.append(('Advantage vs single-feature benchmark', f'Spearman {diff[0]:+.3f} ({ci} {dlo[0]:+.3f} to {dhi[0]:+.3f})',
                     'Model ranks outcomes better than the benchmark' if dlo[0] > 0
                     else 'Benchmark ranks outcomes better' if dhi[0] < 0 else 'No clear difference from the benchmark'))
    psi_value = metrics.get('psi_oot_vs_oof')
    if psi_value is not None and psi_value > s.psi_warn:
        rows.append(('Score shift, training → holdout (PSI)', f'{psi_value:.2f}',
                     f'Above the {s.psi_warn:g} warning level: check the holdout extract for feature changes'))
    scorecard = pd.DataFrame(rows, columns=['Measure', 'Result', 'Interpretation'])
    print('\nValidation scorecard')
    _show(scorecard)
    print(f'Benchmark: {rule_text}.')
    scorecard.to_csv(out / 'validation_scorecard.csv', index=False)

    # Targeting performance and outcome capture (additive outcomes, higher priority only)
    if targeting:
        reached = d['capture'] * total
        pd.DataFrame({
            'audience_pct': d['depth'] * 100, 'donors_selected': d['n_top'], 'outcome_among_selected': reached,
            'outcome_captured_pct': d['capture'] * 100, 'outcome_at_random': d['n_top'] / n * total,
            'outcome_rate_selected': reached / d['n_top'], 'lift': d['lift'], 'lift_lo': d['lift_lo'], 'lift_hi': d['lift_hi'],
            'benchmark_captured_pct': capture_at(y, rule_score, depths, s.seed) * 100,
            'advantage_pts': diff * 100, 'advantage_lo': dlo * 100, 'advantage_hi': dhi * 100,
        }).to_csv(out / 'targeting_performance.csv', index=False)
        weak = (d['lift_lo'] <= 1).to_numpy()
        print('\nTargeting performance')
        _show(pd.DataFrame({
            'Audience selected': [f'Top {v:.0%}' for v in d['depth']],
            'Donors selected': [f'{v:,}' for v in d['n_top']],
            'Observed outcome among selected': [f'{v:,.0f}' if s.is_clf else _num(v, money) for v in reached],
            'Outcome captured': [_pct(v) for v in d['capture']],
            'Lift vs random': [f'{v:.2f}×' + (' †' if w else '') for v, w in zip(d['lift'], weak)],
            'Advantage vs benchmark': [f'{100 * v:+.2f} pts' for v in diff],
        }))
        if weak.any():
            print(f'† {ci} for lift includes 1×: not distinguishable from random at this depth.')

        grid = np.linspace(0.01, 1, 100)
        fig, ax = plt.subplots(figsize=(7, 4.4))
        ax.plot(grid * 100, capture_at(y, prio, grid, s.seed) * 100, label='Model')
        ax.plot(grid * 100, capture_at(y, rule_score, grid, s.seed) * 100, label=f'Benchmark: {rule_text}')
        ax.plot([0, 100], [0, 100], ls='--', c='grey', label='Random')
        ax.errorbar(d['depth'] * 100, d['capture'] * 100, fmt='o', capsize=3, c='C0',
                    yerr=[(d['capture'] - d['capture_lo']) * 100, (d['capture_hi'] - d['capture']) * 100])
        ax.set(xlabel=f'Audience selected (% of donors, {prio_text})', ylabel='Outcome captured (%)',
               title=f'Outcome capture by audience size\n{title}')
        ax.legend(fontsize=8)
        _save_show(fig, out / 'outcome_capture.png')

    dec = decile_table(y, prio, s)
    fig, ax = plt.subplots(figsize=(7, 4.4))
    ax.bar(dec['bin'], dec['event_rate' if s.is_clf else 'mean'])
    ax.axhline(y.mean(), ls='--', c='grey', label='Overall')
    if s.is_clf:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    elif money:
        ax.yaxis.set_major_formatter(lambda v, _: _num(v, True))
    ax.set(xlabel=f'Score decile (1 = {prio_text})', ylabel=outcome_word, xticks=dec['bin'],
           title=f'Observed outcome by score decile\n{title}')
    ax.legend()
    _save_show(fig, out / 'outcome_by_decile.png')

    # Performance by donor segment: every cohort in cohorts.yaml scored with this model; the CSV keeps all of them
    seg_rows, masks, flat = [], {}, hold.reset_index()
    for seg in cohorts:
        try:
            mask = cohort_mask(flat, seg, cohorts).to_numpy()
        except (KeyError, ValueError):     # a filter column this data doesn't have
            continue
        if not mask.any():
            continue
        ys, ss, ps = y[mask], score[mask], prio[mask]
        positives = int((ys != 0).sum())
        enough = (positives if targeting else ys.size) >= MIN_SEGMENT_EVENTS and (not s.is_clf or positives < ys.size)
        row = {'segment': seg, 'donors': ys.size, 'outcome_positive': positives, 'outcome_mean': ys.mean(),
               'ranking': rank_fn(ys, ss) if enough else np.nan}
        if targeting:
            row['lift_top'] = capture_at(ys, ps, [focus], s.seed)[0] / (_top_k(focus, ys.size) / ys.size) if enough else np.nan
        seg_rows.append(row)
        masks[seg] = mask
    seg = pd.DataFrame(seg_rows).sort_values('donors', ascending=False, ignore_index=True)
    # Display the largest judgeable segments, skipping any that overlap 80%+ with this cohort or a larger one shown
    shown, kept = [], [np.ones(n, dtype=bool)]
    for name in seg.loc[seg['ranking'].notna(), 'segment']:
        m = masks[name]
        if all((m & k).sum() / (m | k).sum() < 0.8 for k in kept):
            shown.append(name)
            kept.append(m)
        if len(shown) == MAX_SEGMENTS:
            break
    seg['displayed'] = seg['segment'].isin(shown)
    seg.to_csv(out / 'segment_performance.csv', index=False)
    view_rows = seg[seg['displayed']]
    view = {'Segment': [_cohort_name(v) for v in view_rows['segment']], 'Donors': [f'{v:,}' for v in view_rows['donors']]}
    if positive_word:
        view[positive_word.capitalize()] = [f'{v:,}' for v in view_rows['outcome_positive']]
    view |= {outcome_word: [fmt_outcome(v) for v in view_rows['outcome_mean']],
             rank_name: [f'{v:.3f}' for v in view_rows['ranking']]}
    if targeting:
        view[f'Lift, top {focus:.0%}'] = [f'{v:.2f}×' for v in view_rows['lift_top']]
    print('\nPerformance by donor segment')
    _show(pd.DataFrame(view))
    notes = [f'Showing {len(shown)} of {len(seg)} segments: the largest with at least {MIN_SEGMENT_EVENTS} '
             f"{positive_word or 'donors'}, skipping any that overlap 80% or more with this cohort or a larger segment "
             'shown. segment_performance.csv lists all of them.',
             f"Every segment is scored with this cohort's model ({_cohort_name(cohort)}); no model is trained per segment."]
    if targeting:
        notes.append(f'Lift is recalculated within each segment: its own top {focus:.0%} by score against its own '
                     + ('outcome rate, so it cannot exceed 1 ÷ that rate.' if s.is_clf else 'average outcome.'))
    print('\n'.join(notes))

    # Characteristics of selected donors: observed outcome plus up to 5 strongest drivers that differ between the groups
    ex = pd.read_csv(run_dir / 'explain.csv')
    ex['importance'] = ex['coef' if 'coef' in ex else 'gain'].abs()
    ex = ex.sort_values('importance', ascending=False, ignore_index=True)
    top = rank_desc(prio, s.seed) <= _top_k(focus, n)
    chars = [(f'{outcome_word} (observed)', fmt_outcome(y[top].mean()), fmt_outcome(y[~top].mean()), y[top].mean(), y[~top].mean())]
    for feat in ex['feature']:
        if len(chars) > 5:
            break
        v = _feature_values(hold, feat)
        if v is None:
            continue
        v = v.to_numpy()
        binary = set(np.unique(v[~np.isnan(v)])) <= {0.0, 1.0}
        stat = np.nanmean if binary else np.nanmedian
        a, b = stat(v[top]), stat(v[~top])
        if not (np.isfinite(a) and np.isfinite(b)) or np.isclose(a, b, rtol=0.01, atol=1e-12):
            continue                                          # no visible difference (e.g. 0 vs 0)
        fmt = _pct if binary else _num
        chars.append((f"{_pretty(feat)} ({'share' if binary else 'median'})", fmt(a), fmt(b), a, b))
    who = 'selected donors' if targeting else 'the highest-priority donors'
    group = f'Top {focus:.0%} ({prio_text})'
    print(f'\nCharacteristics of {who}')
    _show(pd.DataFrame([c[:3] for c in chars], columns=['Characteristic', group, 'Everyone else']))
    print("Features are the model's strongest drivers that differ visibly between the groups; "
          'medians for numeric features, shares for yes/no features.')
    pd.DataFrame([(c[0], c[3], c[4]) for c in chars], columns=['characteristic', 'top_group', 'everyone_else']).to_csv(
        out / 'selected_donor_characteristics.csv', index=False)

    # Factors associated with model scores (chart only; values saved to CSV)
    drv = ex.head(10).copy()
    drv['relative_importance'] = 100 * drv['importance'] / drv['importance'].max()
    drv['association'] = [np.sign(pd.Series(v.to_numpy()).corr(pd.Series(score), method='spearman'))
                          if (v := _feature_values(hold, f)) is not None else np.nan for f in drv['feature']]
    drv[['feature', 'relative_importance', 'association']].to_csv(out / 'score_factors.csv', index=False)
    plot = drv.iloc[::-1]
    fig, ax = plt.subplots(figsize=(7.5, 0.34 * len(plot) + 1.4))
    ax.barh([_pretty(f) for f in plot['feature']], plot['relative_importance'],
            color=['C0' if v > 0 else 'C3' if v < 0 else 'grey' for v in plot['association']])
    ax.legend(handles=[Patch(color='C0', label='Associated with higher scores'),
                       Patch(color='C3', label='Associated with lower scores')], loc='lower right', fontsize=8)
    ax.set(xlabel='Relative importance (top = 100)', title=f'Factors associated with model scores\n{title}')
    _save_show(fig, out / 'score_factors.png')
    print('Relative importance is normalized within this model (top = 100), so it is not comparable across model '
          "families. Association is the sign of each factor's rank correlation with the score, not a causal or "
          'independent effect.' + (' For this label, lower scores are the priority.' if lower else ''))
    print(f'Saved to {out}')


validated = [r for summ in summaries for r in summ.to_dict('records')
             if r.get('status') == 'published' and r.get('mode') == 'validation']
if not validated:
    print('No cohorts in validation mode (holdout labels not observed yet): no validation report.')
for v in validated:
    try:
        s_v, _ = settings_from_config(modeling, label=v['label'], task_type=v['task_type'], seed=SEED)
        validation_report(v['label'], v['cohort'], s_v, OUTPUT_DIR / 'validation' / v['label'] / v['cohort'],
                          train_df[cohort_mask(train_df, v['cohort'], cohorts)], oot_df[cohort_mask(oot_df, v['cohort'], cohorts)],
                          BASELINE_RULES.get(v['label']), PRIORITY.get(v['label'], 'higher'))
    except Exception:
        log.exception('%s / %s: validation report failed', v['label'], v['cohort'])

## 7. Run summary

One row per label and cohort. Raises at the end if any label or cohort failed.

In [ ]:
summary = pd.concat(summaries, ignore_index=True)
print(summary.round(4).to_string(index=False))

failed = summary[summary['status'] == 'failed']
if len(failed):
    where = [f"{r.label} / {r.cohort if isinstance(r.cohort, str) else 'all cohorts'}" for r in failed.itertuples()]
    raise RuntimeError(f'{len(failed)} failure(s): {where}; see the log and run_summary.csv')

## 8. Handoff

For each published cohort, `model_spec.json` records `model_version` (a hash of `model.pkl`), the feature list, the input file hashes, CV and validation results, and the library versions in `environment`. `model.pkl` holds only sklearn/LightGBM objects; to reuse it outside this script, install those versions, then:

```python
spec = json.loads((run_dir / 'model_spec.json').read_text())
model = pickle.loads((run_dir / 'model.pkl').read_bytes())
features = scoring_frame[spec['feature_list']]
scores = model.predict_proba(features)[:, 1]   # model.predict(features) for regression labels
```

The scoring frame must be built the way the training frame was: the same as-of-T feature logic, ids normalized as text, and bool columns as 0/1.